# YouTube AI News Explorer

## Introduction

This project demonstrates the use of Generative AI to assist users in exploring current news topics by searching for relevant YouTube videos and providing informative summaries. The system leverages Gemini 2.0 Flash and controls model output through a structured prompt, enabling reliable extraction and evaluation of video content using subtitles.

### Problem Statement

> Today, it is difficult to navigate the overwhelming and often emotionally charged landscape of news — especially in video format.  
> The tone and framing of news can distort a user’s perception and influence critical decisions.

The proposed **AI News Explorer** addresses this challenge by:
- Simplifying information intake through **summarization** and **fact extraction**;
- Providing an **independent assessment** of the **emotional tone** and **agenda** of the content;
- Helping users build their own informed and balanced perspective.

### Key Features

- 🔍 Search YouTube for videos on a given news topic;
- 📝 Download subtitles and generate concise summaries in the user’s language;
- 📌 Extract verifiable facts from the subtitles;
- 🎯 Evaluate:
  - Emotional or judgmental language,
  - Presence of an agenda or strong framing,
  - Whether the video is opinion-based, analytical, or a news briefing;
- ❓ Answer user questions based only on retrieved content;
- 🌐 Multilingual support;
- ⚙️ Powered by:
  - **Gemini 2.0 Flash**,
  - **Function calling**,
  - **Structured function output (JSON)**,
  - **Prompt engineering for output control**.

### How to Run the Notebook

To run this notebook:
- Please ensure you have added your YouTube API key, Google Gemini API key, and Google Application Credentials to Kaggle Secrets under the following names:
  - YOUTUBE_API_KEY
  - GEMINI_API_KEY
  - GOOGLE_APPLICATION_CREDENTIALS_JSON  
- If you're running the notebook in a different environment (e.g. Colab, Jupyter), make sure to include code for defining the required variables with the listed API keys manually.

### Setup: Install and import libraries, load API keys, and define global variables

In [1]:
# Install Libraries

!pip uninstall -qqy jupyterlab  # Remove unused conflicting packages
!pip install -U -q "google-genai==1.7.0"

!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 3.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.5 MB/s eta 0:00:0000:0100:01


In [10]:
import os
import json

# Youtube API libraries
from youtube_transcript_api import YouTubeTranscriptApi
import googleapiclient.discovery

# Google AI
from google import genai
from google.genai import types
from typing import List, Dict

# Securely Access Secrets
# Please ensure you have added your YouTube and Google Gemini API keys, Google Credentials to Kaggle Secrets.
# If you're running the notebook in a different environment, make sure to include code for defining the required variables with the listed API keys.
from kaggle_secrets import UserSecretsClient
try:
    secrets_client = UserSecretsClient()
    youtube_api_key = secrets_client.get_secret("YOUTUBE_API_KEY")
    genai_api_key = secrets_client.get_secret("GOOGLE_API_KEY")
    google_credentials_json = secrets_client.get_secret("GOOGLE_APPLICATION_CREDENTIALS")
    google_credentials = json.loads(google_credentials_json)

    print("API keys and Google Credentials loaded successfully from Kaggle Secrets.")
except Exception as e:
    print(f"Error loading API keys and Google Credentials from Kaggle Secrets: {e}")
    print("Please ensure you have added your API keys and Google Credentials to Kaggle Secrets.")

API keys and Google Credentials loaded successfully from Kaggle Secrets.


### Auxiliary functions

In [11]:
def search_youtube(topic, max_results=5, language="en", api_key=youtube_api_key):
    """Searches YouTube videos on a given topic"""
    try:
        youtube = googleapiclient.discovery.build("youtube", "v3", developerKey=api_key)
        request = youtube.search().list(
            part="id,snippet",
            maxResults=max_results,
            q=topic,
            type="video",
            order="relevance",
            videoDuration="medium",  # short (> 4 min) or medium (4–20 min),
            relevanceLanguage=language,
            safeSearch = "moderate"       
        )
        response = request.execute()

        videos = []
        for item in response["items"]:
            video_id = item["id"]["videoId"]
            snippet = item["snippet"]
            
            # Request more details about the video to learn about subtitles
            video_request = youtube.videos().list(
                part="contentDetails",
                id=video_id
            )
            video_details = video_request.execute()
            
            # Checking for subtitles in the video
            subtitles_available = False
            subtitle_languages = [] #Список языков
            if "contentDetails" in video_details["items"][0]:
                content_details = video_details["items"][0]["contentDetails"]
                if "caption" in content_details and content_details["caption"] == "true":
                    subtitles_available = True

                    # Get the list of available caption languages using captions().list()
                    try:
                        caption_request = youtube.captions().list(
                            part="snippet",
                            videoId=video_id
                        )
                        caption_response = caption_request.execute()
                        for caption_item in caption_response["items"]:
                            subtitle_languages.append(caption_item["snippet"]["language"])
                    except Exception as e:
                        #print(f"Error getting list of subtitle languages: {e}")
                        pass

            # Collecting information about the video into a dictionary
            video_info = {
                "title": snippet["title"],
                "channel": snippet["channelTitle"],
                "published_at": snippet["publishedAt"],
                "video_url": f"https://www.youtube.com/watch?v={video_id}",
                "video_id": video_id
            }
            videos.append(video_info)

        return videos
        
    except Exception as e:
        #print(f"Error while searching for videos on YouTube: {e}")
        return None

In [12]:
def download_youtube_subtitles(video_id, lang='en'):
    """Downloads subtitles from YouTube video (manual or autogenerated) and returns plain text."""

    transcript_entries = []

    # First try to get custom subtitles
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
        for entry in transcript:
            transcript_entries.append(entry['text'])
    except Exception as e:
        #print(f"Error getting custom subtitles: {e}")
        pass

    # If there are no custom subtitles, try auto-generated ones
    if not transcript_entries:
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            transcript_ag = transcript_list.find_generated_transcript([lang])
            transcript_final = transcript_ag.fetch()
            for entry in transcript_final:
                transcript_entries.append(entry['text'])
        except Exception as e:
            #print(f"Error getting auto-generated subtitles: {e}")
            return None

    full_text = " ".join(transcript_entries)
    return full_text

### Orchestration function called by AI

In [13]:
def search_download(
    topic: str,
    num_videos: int,
    language: str
    ) -> List[Dict]:
    """
    Search YouTube for videos on a given topic, download subtitles, and return a dictionary for each video.

    Args:
        topic (str): The topic to search for.
        num_videos (int): Number of videos to retrieve. Default is 5.
        language (str): Language code for subtitles. Default is 'en'.

    Returns:
        List[Dict]: A list of dictionaries with video info and subtitle text (if available).
    """
    
    if not num_videos:
        num_videos = 5
    if not language:
        language = "en"
    
    try:
        videos_info = search_youtube(topic, max_results=num_videos, language=language)

        if videos_info:
            
            for video in videos_info:
                
                    subtitles = download_youtube_subtitles(video["video_id"], lang=language)
                    if subtitles:
                        video['text'] = subtitles
                        
                    else:
                        #print(f"Failed to download or no subtitles for video: {video['video_url']}")
                        video['text'] = "Failed to download or no subtitles"
               
        else:
            #print("No videos found on this topic")
            pass
        return videos_info
    except Exception as e:
        #print(f"General error: {e}")
        return []

### AI control model

In [14]:
# Define a retry policy. The model might make multiple consecutive calls automatically
# for a complex query, this ensures the client retries if it hits quota limits.
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
  genai.models.Models.generate_content = retry.Retry(
      predicate=is_retriable)(genai.models.Models.generate_content)

In [32]:
# Instructions for AI and the search_download function call parameter

sd_tools = [search_download]

instruction = """
### Your Role

You are a friendly and knowledgeable assistant helping users explore current news topics on YouTube.

When the user sends their first message, detect the language of the message (if it is "hi" or "hello" - english), respond entirely in that language — including greetings, inform them that you can:
- Find recent YouTube videos on any news topic they request;
- Provide a **short summary** for each video in their language;
- Extract **key facts** from subtitles;
- Evaluate **Emotional tone**;  
- Evaluate **Positioning**;  
- Evaluate **Content type**;
- Answer specific questions based on the content.

End your message with:  
**"What topic are you interested in?"**

---

### Video Search and Subtitles

Use the `search_download` function to find up to **5 YouTube videos** related to the user's topic.

**By default:**
- Translate the topic to English before calling the function if the user's language is not English;
- The user may request **up to 10 videos**, but no more.

Each result of calling function includes:
- `title`, `channel`, `published_at`, `video_url`, `video_id`, and `text` (subtitles, if available).

---

### Handling Videos with and without Subtitles

#### 🔴 If **no subtitles** are found for any video:
- Respond: *"Unfortunately, subtitles could not be retrieved for the videos found."*
- Offer two options:
  - Show a list of video titles, channels, and links;
  - Retry the search with up to 10 videos.

#### 🟢 If **subtitles are available**:
Generate a structured summary block for **each video with subtitles** using the following exact format:
📌 {channel}, {published_at}, {HH:MM}, {video_url} 
📄 {title} 
✍️ {short summary — 3 simple sentences based only on subtitles} 
📊 Fact 1: {verifiable fact from subtitles} 
📊 Fact 2: {the second verifiable fact from subtitles if there is} 
💬 Emotional tone: {Neutral / Mildly emotional / Strongly emotional}  
🎯 Positioning: {None / Present but subtle / Clearly expressed agenda}  
📚 Content type: {News summary / Opinion / Analytical overview}

⚠️ **Important**:
- **Never omit any field**. If you can't find a fact, say so explicitly.
- Always write in the **user’s language**.
- Avoid assumptions or opinions not grounded in the text.

---

### Evaluation Definitions

**Emotional tone**:
- **Neutral**: Purely descriptive, no emotional or subjective language.
- **Mildly emotional**: Occasional emotionally loaded or evaluative words.
- **Strongly emotional**: Frequent or dominant emotional, dramatic, or persuasive language.

**Positioning** (Agenda presence):
- **None**: Balanced or neutral coverage, no clear stance.
- **Present but subtle**: A perspective is visible, but not explicitly pushed.
- **Clearly expressed agenda**: Strong preference or position is promoted throughout.

**Content type**:
- **News summary**: Facts without interpretation, like headlines or event logs.
- **Opinion**: Clear personal stance, often uses first person or rhetorical devices.
- **Analytical overview**: Provides context, reasons, and consequences.

--- 

### Other Rules

- Maintain a neutral, informative, and user-friendly tone.
- Do not output raw subtitles, file paths, logs, or internal errors.
- If subtitles exist but contain **only music, ads, or filler content**, treat the video as **having no usable subtitles**.
- At the end of the summaries, offer:
> *"Would you like to ask questions based on the content?"*
- Answer the user's questions **only based on the subtitles and facts extracted from the video**.  
  If the answer cannot be found in the available content, respond with:  
  *"The video does not contain enough information to answer this question. Would you like to broaden your search or include more videos?"*

"""

client = genai.Client(api_key=genai_api_key)

# Start a chat with automatic function calling enabled.
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=sd_tools,
    ),
)

In [33]:
# Start chating with YouTube News AI Explorer from "Hello!" and then to the topic of interest to you

response = chat.send_message('Hi!')
print(f"\n{response.text}")


Hello! I can help you find recent YouTube videos on any news topic you request. I can also provide a short summary for each video, extract key facts from subtitles, evaluate emotional tone and positioning, determine the content type, and answer specific questions based on the content.

What topic are you interested in?

